Импорт библиотек

In [2]:
!pip install simpy
import simpy
import random
import pandas as pd

описываем класс перекрестка, бизнес-логику процесса

In [3]:
class Intersection:
    def __init__(self, env):
        self.env = env
        # только 1 машина может находиться на перекрестке от каждой улицы
        self.road_a = simpy.Resource(env, capacity=1)
        self.road_c = simpy.Resource(env, capacity=1)
        # состояние светофора (true = зеленый, false = красный для направления C-B)
        self.light_c_b = False
        self.passed_cars = 0

    def traffic_light_control(self):
        while True:
            yield self.env.timeout(20) # переключение каждые 20 сек
            self.light_c_b = not self.light_c_b

Логика движения машин

In [4]:
def car_process(env, name, intersection, road):
    arrival_time = env.now

    # запрос на въезд на перекресток
    with road.request() as req:
        yield req
        # время проезда перекрестка == 2 секунды
        yield env.timeout(2)
        intersection.passed_cars += 1
        print(f"[{env.now:.1f} сек] Машина {name} проехала перекресток.")

интенсивность: $3 \pm 2$ секунды

In [5]:
def stream_a(env, intersection):
    i = 0
    while True:
        yield env.timeout(random.uniform(1, 5))
        i += 1
        env.process(car_process(env, f"A-{i}", intersection, intersection.road_a))

def stream_c(env, intersection):
    i = 0
    while True:
        yield env.timeout(random.uniform(1, 5))
        i += 1
        env.process(car_process(env, f"C-{i}", intersection, intersection.road_c))

Запуск и визуализация результата

In [6]:
# Настройка симуляции
env = simpy.Environment()
intersection = Intersection(env)

# Запуск процессов
env.process(intersection.traffic_light_control())
env.process(stream_a(env, intersection))
env.process(stream_c(env, intersection))

# Запуск на 100 секунд
env.run(until=100)

print("-" * 30)
print(f"Итого проехало машин: {intersection.passed_cars}")

[4.9 сек] Машина C-1 проехала перекресток.
[5.3 сек] Машина A-1 проехала перекресток.
[8.6 сек] Машина C-2 проехала перекресток.
[9.2 сек] Машина A-2 проехала перекресток.
[11.7 сек] Машина C-3 проехала перекресток.
[12.5 сек] Машина A-3 проехала перекресток.
[14.6 сек] Машина C-4 проехала перекресток.
[14.9 сек] Машина A-4 проехала перекресток.
[16.6 сек] Машина C-5 проехала перекресток.
[19.0 сек] Машина C-6 проехала перекресток.
[19.9 сек] Машина A-5 проехала перекресток.
[23.5 сек] Машина C-7 проехала перекресток.
[24.4 сек] Машина A-6 проехала перекресток.
[26.4 сек] Машина C-8 проехала перекресток.
[26.4 сек] Машина A-7 проехала перекресток.
[30.2 сек] Машина C-9 проехала перекресток.
[31.4 сек] Машина A-8 проехала перекресток.
[32.7 сек] Машина C-10 проехала перекресток.
[35.3 сек] Машина A-9 проехала перекресток.
[35.4 сек] Машина C-11 проехала перекресток.
[37.5 сек] Машина A-10 проехала перекресток.
[39.4 сек] Машина C-12 проехала перекресток.
[40.7 сек] Машина A-11 проехала 